# Interpretable Deep Learning for Respiratory Disorder Detection: Colab training

Runs the project's own modules on a Colab GPU: data preparation (download, quality checks, patient-level split, CLAHE preprocessing), baseline CNN + ResNet50 training, test-set evaluation and Grad-CAM. Results match the local CPU pipeline, only faster.

**Runtime -> Change runtime type -> GPU** before running.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

## 2. Get the project code

Either set `REPO_URL` to your git remote, or leave it empty and upload a zip of the project folder (without `data/`, `models/` and `.venv/`).

In [ ]:
import os, zipfile
REPO_URL = ''  # e.g. 'https://github.com/<you>/respiratory-xray.git'
if REPO_URL:
    !git clone -q $REPO_URL project
else:
    from google.colab import files
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall('/content/upload')
    # The project root is the folder that contains src/, wherever it sits in the zip
    root = next(d for d, subdirs, _ in os.walk('/content/upload') if 'src' in subdirs)
    !rm -rf project && cp -r "{root}" project
%cd project

## 3. Install the few packages Colab lacks

Colab already ships TensorFlow, scikit-learn, pandas and matplotlib.

In [ ]:
!pip install -q opencv-python-headless huggingface_hub tabulate

## 4. Data: download, quality checks, patient-level split, CLAHE preprocessing

In [ ]:
!python -m src.data.prepare

## 5. Train the baseline CNN and ResNet50 (5 frozen + 3 fine-tune epochs)

In [ ]:
!python -m src.train --task binary

## 6. Grad-CAM on 10 held-out test images

In [ ]:
!python -m src.gradcam

## 7. Results (held-out test set)

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open('results/model_comparison.md').read()))
for f in ['results/roc_curves.png', 'results/confusion_matrix_resnet50.png',
          'results/training_curves_resnet50.png', 'results/gradcam/gradcam_grid.png']:
    display(Image(f, width=800))

## 8. Download the trained models and results

In [ ]:
!zip -qr trained_outputs.zip models results splits
from google.colab import files
files.download('trained_outputs.zip')